# 몬테카를로 vs Q-러닝 성능 비교 분석 (Cliff Walking)

이 노트북은 절벽 걷기(Cliff Walking) 환경에서 두 가지 강화학습 알고리즘의 성능을 다양한 신경망 구조별로 비교합니다.

## 분석 전략
1. **복수 모델 아키텍처**: Shallow부터 Deep까지 5가지 구조 테스트
2. **알고리즘 비교**: 에피소드 단위 업데이트(MC) vs 스텝 단위 업데이트(QL)
3. **지표**: 누적 보상, 도달 스텝 수, 절벽 추락 횟수

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import Adam
import copy
import tensorflow as tf

## 1. 환경 정의 (Cliff Walking Environment)

In [ ]:
class Environment():
    def __init__(self):
        self.cliff = -100
        self.road = -1
        self.goal = 1
        self.goal_position = [3, 11]
        self.start_position = [3, 0]
        self.reward_list = [
            [self.road]*12,
            [self.road]*12,
            [self.road]*12,
            [self.road] + [self.cliff]*10 + [self.goal]
        ]
        # environment.py와 동일한 로직을 위해 문자열 리스트 추가
        self.reward_list1 = [
            ["road"]*12,
            ["road"]*12,
            ["road"]*12,
            ["road"] + ["cliff"]*10 + ["goal"]
        ]
        self.reward = np.asarray(self.reward_list)

    def move(self, agent, action):
        done = False
        new_pos = agent.pos + agent.action[action]

        # environment.py의 move() 로직과 동일하게 수정
        if self.reward_list1[agent.pos[0]][agent.pos[1]] == "goal":
            reward = self.goal
            observation = agent.set_pos(agent.pos)
            done = True
        elif (new_pos[0] < 0 or new_pos[0] >= 4 or 
              new_pos[1] < 0 or new_pos[1] >= 12 or 
              self.reward_list1[new_pos[0]][new_pos[1]] == self.cliff or
              self.reward_list1[new_pos[0]][new_pos[1]] == "cliff"):
            reward = self.cliff
            observation = agent.set_pos(self.start_position)
            done = True
        else:
            observation = agent.set_pos(new_pos)
            reward = self.reward[observation[0], observation[1]]
            
        return observation, reward, done


## 2. 에이전트 정의 (MC & Q-Learning)

In [ ]:
class MCAgent:
    def __init__(self, state_size=48, action_size=4, learning_rate=0.001, gamma=0.99, hidden_layers=[64, 64]):
        self.state_size, self.action_size = state_size, action_size
        self.gamma, self.learning_rate = gamma, learning_rate
        self.epsilon, self.epsilon_decay, self.epsilon_min = 1.0, 0.995, 0.01
        self.hidden_layers = hidden_layers
        self.pos = [3, 0]
        self.action = np.array([[-1,0],[0,1],[1,0],[0,-1]])
        self.model = self._build_model()
        self.memory = []

    def _build_model(self):
        model = Sequential()
        for i, units in enumerate(self.hidden_layers):
            if i == 0: model.add(Dense(units, input_dim=self.state_size, activation='relu'))
            else: model.add(Dense(units, activation='relu'))
        model.add(Dense(self.action_size, activation='linear'))
        model.compile(loss='mse', optimizer=Adam(lr=self.learning_rate))
        return model

    def set_pos(self, pos): self.pos = np.array(pos); return self.pos
    def state_to_onehot(self, pos): 
        onehot = np.zeros(self.state_size)
        onehot[pos[0]*12 + pos[1]] = 1.0
        return np.reshape(onehot, [1, self.state_size])

    def select_action(self, state):
        if np.random.rand() <= self.epsilon: return np.random.randint(self.action_size)
        q_values = self.model.predict(state, verbose=0)
        return np.argmax(q_values[0])

    def train_model(self):
        G = 0
        states, targets = [], []
        for i in reversed(range(len(self.memory))):
            s, a, r = self.memory[i]
            G = r + self.gamma * G
            s_oh = self.state_to_onehot(s)
            target = self.model.predict(s_oh, verbose=0)
            target[0][a] = G
            states.append(s_oh[0]); targets.append(target[0])
        self.model.fit(np.array(states), np.array(targets), epochs=1, verbose=0)
        self.memory = []
        if self.epsilon > self.epsilon_min: self.epsilon *= self.epsilon_decay

class QLearningAgent:
    def __init__(self, state_size=48, action_size=4, learning_rate=0.001, gamma=0.99, hidden_layers=[64, 64]):
        self.state_size, self.action_size = state_size, action_size
        self.gamma, self.learning_rate = gamma, learning_rate
        self.epsilon, self.epsilon_decay, self.epsilon_min = 1.0, 0.995, 0.01
        self.hidden_layers = hidden_layers
        self.pos = [3, 0]
        self.action = np.array([[-1,0],[0,1],[1,0],[0,-1]])
        self.model = self._build_model()

    def _build_model(self):
        model = Sequential()
        for i, units in enumerate(self.hidden_layers):
            if i == 0: model.add(Dense(units, input_dim=self.state_size, activation='relu'))
            else: model.add(Dense(units, activation='relu'))
        model.add(Dense(self.action_size, activation='linear'))
        model.compile(loss='mse', optimizer=Adam(lr=self.learning_rate))
        return model

    def set_pos(self, pos): self.pos = np.array(pos); return self.pos
    def state_to_onehot(self, pos): 
        onehot = np.zeros(self.state_size)
        onehot[pos[0]*12 + pos[1]] = 1.0
        return np.reshape(onehot, [1, self.state_size])

    def select_action(self, state):
        if np.random.rand() <= self.epsilon: return np.random.randint(self.action_size)
        q_values = self.model.predict(state, verbose=0)
        return np.argmax(q_values[0])

    def train_model(self, s, a, r, ns, done):
        s_oh, ns_oh = self.state_to_onehot(s), self.state_to_onehot(ns)
        target = self.model.predict(s_oh, verbose=0)
        if done: 
            target[0][a] = r
        else: 
            target[0][a] = r + self.gamma * np.max(self.model.predict(ns_oh, verbose=0)[0])
        self.model.fit(s_oh, target, epochs=1, verbose=0)
        if done and self.epsilon > self.epsilon_min: self.epsilon *= self.epsilon_decay

## 3. 실험 실행 루프

In [ ]:
def run_experiment(algo, episodes, layers):
    env = Environment()
    agent = MCAgent(hidden_layers=layers) if algo == 'MC' else QLearningAgent(hidden_layers=layers)
    rewards, steps, falls = [], [], 0
    
    for e in tqdm(range(episodes), desc=f"{algo} {layers}"):
        state = env.start_position
        agent.set_pos(state)
        total_r, step, done = 0, 0, False
        while not done:
            s_oh = agent.state_to_onehot(state)
            action = agent.select_action(s_oh)
            next_state, reward, done = env.move(agent, action)
            if algo == 'MC': agent.memory.append((state, action, reward))
            else: agent.train_model(state, action, reward, next_state, done)
            state = next_state
            total_r += reward; step += 1
            if reward == env.cliff: falls += 1
            if step > 200: done = True
        if algo == 'MC': agent.train_model()
        rewards.append(total_r); steps.append(step)
    return rewards, steps, falls

EPISODES = 300
architectures = {
    "Shallow-Tiny [32]": [32],
    "Shallow-Mid [32, 32]": [32, 32],
    "Standard [64, 64]": [64, 64],
    "Deep-Slim [32, 32, 32]": [32, 32, 32],
    "Deep-Large [64, 64, 64]": [64, 64, 64]
}

all_results = []
num_arch = len(architectures)
plt.figure(figsize=(18, 5 * num_arch))

for i, (name, layers) in enumerate(architectures.items()):
    print(f"Testing: {name}...")
    mc_r, mc_s, mc_f = run_experiment('MC', EPISODES, layers)
    ql_r, q_s, ql_f = run_experiment('QL', EPISODES, layers)
    
    all_results.append([name, "MC", np.mean(mc_r[-50:]), np.mean(mc_s[-50:]), mc_f])
    all_results.append([name, "QL", np.mean(ql_r[-50:]), np.mean(q_s[-50:]), ql_f])
    
    plt.subplot(num_arch, 2, i*2 + 1)
    plt.plot(mc_r, label='MC'); plt.plot(ql_r, label='QL')
    plt.title(f"Reward - {name}"); plt.legend()
    
    plt.subplot(num_arch, 2, i*2 + 2)
    plt.plot(mc_s, label='MC'); plt.plot(q_s, label='QL')
    plt.title(f"Steps - {name}"); plt.legend()

plt.tight_layout(); plt.show()

## 4. 결과 요약 표

In [ ]:
print("="*85)
print(f"{'Architecture':<20} | {'Algo':<4} | {'Avg Reward (Last 50)':<20} | {'Avg Steps':<10} | {'Falls':<5}")
print("-"*85)
for r in all_results:
    print(f"{r[0]:<20} | {r[1]:<4} | {r[2]:<20.2f} | {r[3]:<10.2f} | {r[4]:<5}")
print("="*85)